## Multi-protocol CNN Raw CSI Experiments

This notebook runs the fixed-capacity CNN across block, LOVO, and cross-session protocols with one independent run per configured seed.

In [ ]:
from __future__ import annotations

import pandas as pd
import torch

from utils.config import (
    ANCHOR_GROUPS,
    ARCHITECTURE,
    BANDS_TO_RUN,
    DATA_DIR,
    DEFAULT_CNN_PARAMS,
    DEFAULT_OVERLAP_SIZE,
    DEFAULT_WINDOW_SIZE,
    EXPECTED_ANCHORS,
    EXPECTED_SUBCARRIERS,
    SEEDS,
)
from utils.DL.dl_pipeline import (
    create_position_label_encoder,
    dl_display_model_names,
    dl_per_room_table,
    dl_seed_summary_table,
    get_cache_path,
    get_results_path,
    load_all_dl_predictions,
    prepare_dl_data,
    print_torch_environment,
    run_dl_experiments,
)
from utils.plots import (
    plot_dl_block_vs_lovo_metrics,
    plot_dl_cdf_comparison,
    plot_dl_spatial_generalization,
    plot_dl_volunteer_variability,
)

### Configuration

In [2]:
CALIBRATION_MODE = "rssi"    # ("none", "packet_norm", "rssi")

CSV_PROCESSING_OPTIONS = {
    "max_workers": 1,
    "cache_dir": None,
    "use_cache": True,
    "force_reprocess": False,
    "min_rssi_dbm": -95.0,
}

MAGNITUDE_PROCESSING_OPTIONS = {
    "normalization": "empty_baseline",  # none | zscore | minmax | packet_minmax | empty_baseline
    "epsilon": 1e-8,
}
NORMALIZATION_BASELINE_SCOPE = "per_session"

FEATURE_EXTRACTION_OPTIONS = {
    "window_size": DEFAULT_WINDOW_SIZE,
    "overlap_size": DEFAULT_OVERLAP_SIZE,
    "require_all_esps": False,
}

BLOCK_COUNT = 10
TEST_SIZE = 0.30
VALIDATION_SIZE = 0.15
RANDOM_STATE = 42
FORCE_RETRAIN = False
SPLIT_MODES = ("block", "lovo")

RUN_CLASSIFICATION = True

ANALYSIS_SEED = SEEDS[0]
ANALYSIS_MODEL = "CNN_room" if ARCHITECTURE == "room_stacked" else "CNN"
ANALYSIS_BANDS = ("Fusion",) if ARCHITECTURE == "room_stacked" else BANDS_TO_RUN
CONFUSION_DATASET = "Fusion"
CONFUSION_SPLIT = "block" if "block" in SPLIT_MODES else SPLIT_MODES[0]
SHOW_ARCHITECTURE_COMPARISON = True
SHOW_CDF_BY_BAND = True
SHOW_CDF_BY_MODEL = True
SHOW_BOXPLOT = True
SHOW_FLOOR_PLAN = True
SHOW_CONFUSION_MATRICES = True
SHOW_PER_ROOM_PLOTS = False


CNN_PARAMS = {
    **DEFAULT_CNN_PARAMS,
    "model_label": "CNN",
    "epochs": 70,
    "patience": 15,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "validation_size": VALIDATION_SIZE,
    "n_blocks": BLOCK_COUNT,
    "anchor_groups": ANCHOR_GROUPS,
    "torch_version": torch.__version__,
}

preproc_opts = dict(MAGNITUDE_PROCESSING_OPTIONS)
preproc_opts["baseline_scope"] = NORMALIZATION_BASELINE_SCOPE
feat_opts = dict(FEATURE_EXTRACTION_OPTIONS)

feature_cache_dir = get_cache_path(preproc_opts, feat_opts)
results_dir = get_results_path()
plots_dir = results_dir / "plots"
tables_dir = results_dir / "tables"
for directory in (plots_dir, results_dir / "predictions", tables_dir):
    directory.mkdir(parents=True, exist_ok=True)


def _slugify(value: str) -> str:
    """Convert a display value to a compact filename-safe slug."""
    return value.lower().replace(".", "-").replace(" ", "-").strip("-")


analysis_plots_dir = (
    plots_dir / f"analysis_{_slugify(ANALYSIS_MODEL)}_seed-{ANALYSIS_SEED}"
)
analysis_plots_dir.mkdir(parents=True, exist_ok=True)

print(f"Feature cache path: {feature_cache_dir}")
print(f"Results path: {results_dir}")
print(f"Analysis figures path: {analysis_plots_dir}")


Feature cache path: /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/.cache/dataframes/preproc=norm-empty_baseline_scope-per_session/feat=win60-step0
Results path: /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results
Analysis figures path: /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/plots/analysis_cnn_room_seed-42


### Environment

In [3]:
DEVICE = print_torch_environment(require_cuda=True)


torch.__version__: 2.11.0+cu128
torch.version.cuda: 12.8
torch.cuda.get_device_name(0): NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition
torch.cuda.get_device_capability(0): (12, 0)
CUDA matmul smoke test result: [[0.0, 1.0, 2.0, 3.0], [4.0, 5.0, 6.0, 7.0], [8.0, 9.0, 10.0, 11.0], [12.0, 13.0, 14.0, 15.0]] (PASS)


### Data

In [4]:
processed_magnitude_data, feature_dataframes, csv_diagnostics, magnitude_summary = prepare_dl_data(
    DATA_DIR,
    calibration_mode=CALIBRATION_MODE,
    csv_options=CSV_PROCESSING_OPTIONS,
    preproc_opts=preproc_opts,
    feat_opts=feat_opts,
)
display(magnitude_summary.head())


Scenarios present: 1
Locations: 53 | Users: 6 | ESPs: 19
[Z-0 inventory]
  total Z-0 files found: 171
  Z-0 count per (user, trial):
    (user=01, trial=01): 19
    (user=01, trial=02): 19
    (user=02, trial=01): 19
    (user=03, trial=01): 19
    (user=03, trial=02): 19
    (user=04, trial=01): 19
    (user=05, trial=01): 19
    (user=05, trial=02): 19
    (user=06, trial=01): 19
  breakdown by user / esp / trial:
    user=01 esp=01 trial=01: 1
    user=01 esp=01 trial=02: 1
    user=01 esp=02 trial=01: 1
    user=01 esp=02 trial=02: 1
    user=01 esp=03 trial=01: 1
    user=01 esp=03 trial=02: 1
    user=01 esp=04 trial=01: 1
    user=01 esp=04 trial=02: 1
    user=01 esp=05 trial=01: 1
    user=01 esp=05 trial=02: 1
    user=01 esp=07 trial=01: 1
    user=01 esp=07 trial=02: 1
    user=01 esp=08 trial=01: 1
    user=01 esp=08 trial=02: 1
    user=01 esp=09 trial=01: 1
    user=01 esp=09 trial=02: 1
    user=01 esp=10 trial=01: 1
    user=01 esp=10 trial=02: 1
    user=01 esp=11 tri

,scenario,location,user,esp,trial,samples,subcarriers,normalization,baseline_scope
0,1,C-1,06,16,01,2088,56,empty_baseline,per_session
1,1,C-1,06,15,01,1942,56,empty_baseline,per_session
2,1,C-1,06,09,01,1689,50,empty_baseline,per_session
3,1,C-1,06,07,01,1965,50,empty_baseline,per_session
4,1,C-1,06,04,01,1913,50,empty_baseline,per_session


In [5]:
for band in BANDS_TO_RUN:
    df = feature_dataframes[band]
    print(f"{band}: {df.shape[0]} windows, {df.shape[1]} columns")
    print(f"{band} dataframe hash: {pd.util.hash_pandas_object(df, index=True).sum()}")


2.4 GHz: 13588 windows, 2708 columns
2.4 GHz dataframe hash: 11876030231112758796
5 GHz: 14478 windows, 3368 columns
5 GHz dataframe hash: 1871796993234382690
Fusion: 13568 windows, 6068 columns
Fusion dataframe hash: 18372045827207501857


In [6]:
label_encoder = create_position_label_encoder(
    feature_dataframes,
    results_dir=results_dir,
    expected_classes=52,
)
print(label_encoder.classes_)


[CNN] label classes saved to /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/manifests/cnn_label_classes.json
['A-1' 'A-13' 'A-14' 'A-2' 'A-5' 'B-1' 'B-10' 'B-11' 'B-12' 'B-13' 'B-14'
 'B-2' 'B-5' 'B-8' 'C-1' 'C-10' 'C-11' 'C-14' 'C-2' 'C-3' 'C-4' 'C-5'
 'C-6' 'C-7' 'C-8' 'C-9' 'D-1' 'D-2' 'D-3' 'D-4' 'D-5' 'D-6' 'D-7' 'D-8'
 'E-1' 'E-10' 'E-11' 'E-12' 'E-13' 'E-2' 'E-3' 'E-4' 'E-5' 'E-6' 'E-7'
 'E-8' 'F-10' 'F-11' 'F-12' 'F-13' 'F-5' 'F-8']


### Train And Evaluate

In [7]:
if RUN_CLASSIFICATION:
    cnn_runs = run_dl_experiments(
        processed_magnitude_data,
        feature_dataframes,
        bands=BANDS_TO_RUN,
        split_modes=SPLIT_MODES,
        label_encoder=label_encoder,
        device=DEVICE,
        results_dir=results_dir,
        plots_dir=plots_dir,
        params=CNN_PARAMS,
        preproc_opts=preproc_opts,
        feat_opts=feat_opts,
        expected_subcarriers=EXPECTED_SUBCARRIERS,
        expected_anchors=EXPECTED_ANCHORS,
        architecture=ARCHITECTURE,
        seeds=SEEDS,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        n_blocks=BLOCK_COUNT,
        val_size=VALIDATION_SIZE,
        force_retrain=FORCE_RETRAIN,
    )
else:
    print("Classification skipped because RUN_CLASSIFICATION is False.")


=== Fusion ===
[window arrays] cache path: /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/.cache/dataframes/preproc=norm-empty_baseline_scope-per_session/feat=win60-step0/window_arrays/fusion
[window arrays cache hit] /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/.cache/dataframes/preproc=norm-empty_baseline_scope-per_session/feat=win60-step0/window_arrays/fusion
[window arrays] 2.4 GHz: shape=(13568, 9, 50, 60), dtype=float16
[window arrays] 2.4 GHz anchors: ['esp_01', 'esp_02', 'esp_03', 'esp_04', 'esp_05', 'esp_07', 'esp_08', 'esp_09', 'esp_10']
[window arrays] 5 GHz: shape=(13568, 10, 56, 60), dtype=float16
[window arrays] 5 GHz anchors: ['esp_11', 'esp_12', 'esp_13', 'esp_14', 'esp_15', 'esp_16', 'esp_17', 'esp_18', 'esp_19', 'esp_20']
[room window arrays] padding rows=2, derived from first-conv kernel_size - 1 (3 - 1).
[room window arrays] cache path: /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/.cache/dataframes/preproc=norm-empty_baseline_scope-per_session

/home/pedro.monteiro@co.it.pt/Desktop/thesis-project/utils/DL/dl_pipeline.py:329: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  band: torch.as_tensor(array[array_index], dtype=torch.float32)
/home/pedro.monteiro@co.it.pt/Desktop/thesis-project/utils/DL/dl_pipeline.py:329: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Tr

[CNN_room] Fusion/block fold=single epoch 01/70: train_loss=3.8937 train_acc=0.0573 val_loss=3.9063 val_acc=0.0467 seconds=4.0
[CNN_room] Fusion/block fold=single epoch 02/70: train_loss=3.6187 train_acc=0.1229 val_loss=3.6425 val_acc=0.0733 seconds=0.5
[CNN_room] Fusion/block fold=single epoch 03/70: train_loss=3.2350 train_acc=0.1575 val_loss=3.2276 val_acc=0.1600 seconds=0.5
[CNN_room] Fusion/block fold=single epoch 04/70: train_loss=2.9734 train_acc=0.1832 val_loss=2.9753 val_acc=0.1800 seconds=0.5
[CNN_room] Fusion/block fold=single epoch 05/70: train_loss=2.7877 train_acc=0.2148 val_loss=2.8251 val_acc=0.1933 seconds=0.5
[CNN_room] Fusion/block fold=single epoch 06/70: train_loss=2.6489 train_acc=0.2384 val_loss=2.7380 val_acc=0.2200 seconds=0.5
[CNN_room] Fusion/block fold=single epoch 07/70: train_loss=2.5272 train_acc=0.2671 val_loss=2.5894 val_acc=0.2800 seconds=0.5
[CNN_room] Fusion/block fold=single epoch 08/70: train_loss=2.3984 train_acc=0.3067 val_loss=2.4957 val_acc=0.2

/home/pedro.monteiro@co.it.pt/Desktop/thesis-project/utils/DL/dl_pipeline.py:329: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  band: torch.as_tensor(array[array_index], dtype=torch.float32)
/home/pedro.monteiro@co.it.pt/Desktop/thesis-project/utils/DL/dl_pipeline.py:329: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Tr

[predictions] Saved /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/predictions/dl__cnn_room__fusion__block__ebl-session__s42__dc4ee3.parquet
[plots] saved /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/plots/dl__cnn_room__fusion__block__ebl-session__s42__dc4ee3/training_curves.png at dpi=200. PNG is raster; set PLOT_FORMAT='pdf' for a LaTeX vector figure without other code changes.
[DL GPU] run_id=dl__cnn_room__fusion__block__ebl-session__s42__dc4ee3 peak_cuda_memory_bytes=2685729792
[results upsert] /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/runs.csv: 46 row(s)
Seeds: random=43, numpy=43, torch=43
[trial filter] split=block trials=['01'] kept=8889/13568
[protocol] split=block trials_used=['01'] n_train=4934 n_test=1388 users=['01', '02', '03', '04', '05', '06'] train_users=['01', '02', '03', '04', '05', '06'] test_users=['01', '02', '03', '04', '05', '06']
[trial filter] split=block trials=['01'] kept=4934/4934
[protocol] split=block trial

...

[predictions] Saved /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/predictions/dl__cnn_room__fusion__lovo__ebl-session__s44__5ccebc__fold-05.parquet
[plots] saved /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/plots/dl__cnn_room__fusion__lovo__ebl-session__s44__5ccebc/training_curves__fold-05.png at dpi=200. PNG is raster; set PLOT_FORMAT='pdf' for a LaTeX vector figure without other code changes.
[DL LOVO] held_out_user=06 validation_user=05 (entire user, deterministic rotation)
[CNN_room] Fusion/lovo fold=06 parameters=195636
[DataLoader] device=cuda settings={'batch_size': 256, 'num_workers': 8, 'pin_memory': True, 'persistent_workers': True, 'prefetch_factor': 4}
[DataLoader] device=cuda settings={'batch_size': 256, 'num_workers': 8, 'pin_memory': True, 'persistent_workers': True, 'prefetch_factor': 4}
[DataLoader] device=cuda settings={'batch_size': 256, 'num_workers': 8, 'pin_memory': True, 'persistent_workers': True, 'prefetch_factor': 4}


/home/pedro.monteiro@co.it.pt/Desktop/thesis-project/utils/DL/dl_pipeline.py:329: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  band: torch.as_tensor(array[array_index], dtype=torch.float32)
/home/pedro.monteiro@co.it.pt/Desktop/thesis-project/utils/DL/dl_pipeline.py:329: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Tr

[CNN_room] Fusion/lovo fold=06 epoch 01/70: train_loss=3.7288 train_acc=0.0762 val_loss=3.9627 val_acc=0.0215 seconds=3.3
[CNN_room] Fusion/lovo fold=06 epoch 02/70: train_loss=3.0692 train_acc=0.1552 val_loss=3.9653 val_acc=0.0983 seconds=1.0
[CNN_room] Fusion/lovo fold=06 epoch 03/70: train_loss=2.7240 train_acc=0.2203 val_loss=3.5151 val_acc=0.1857 seconds=1.0
[CNN_room] Fusion/lovo fold=06 epoch 04/70: train_loss=2.5174 train_acc=0.2671 val_loss=3.2681 val_acc=0.2059 seconds=1.0
[CNN_room] Fusion/lovo fold=06 epoch 05/70: train_loss=2.3456 train_acc=0.3042 val_loss=3.1425 val_acc=0.2079 seconds=1.1
[CNN_room] Fusion/lovo fold=06 epoch 06/70: train_loss=2.2036 train_acc=0.3415 val_loss=3.3537 val_acc=0.1750 seconds=1.1
[CNN_room] Fusion/lovo fold=06 epoch 07/70: train_loss=2.0877 train_acc=0.3759 val_loss=3.1619 val_acc=0.2059 seconds=1.0
[CNN_room] Fusion/lovo fold=06 epoch 08/70: train_loss=1.9823 train_acc=0.4010 val_loss=3.0710 val_acc=0.2288 seconds=1.0
[CNN_room] Fusion/lovo f

/home/pedro.monteiro@co.it.pt/Desktop/thesis-project/utils/DL/dl_pipeline.py:329: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  band: torch.as_tensor(array[array_index], dtype=torch.float32)
/home/pedro.monteiro@co.it.pt/Desktop/thesis-project/utils/DL/dl_pipeline.py:329: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Tr

[predictions] Saved /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/predictions/dl__cnn_room__fusion__lovo__ebl-session__s44__5ccebc__fold-06.parquet
[plots] saved /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/plots/dl__cnn_room__fusion__lovo__ebl-session__s44__5ccebc/training_curves__fold-06.png at dpi=200. PNG is raster; set PLOT_FORMAT='pdf' for a LaTeX vector figure without other code changes.
[predictions] Saved /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/predictions/dl__cnn_room__fusion__lovo__ebl-session__s44__5ccebc.parquet
[results upsert] /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/runs_folds.csv: 72 row(s)
[DL LOVO] seed=44 fold mean position_accuracy=0.1151 +/- 0.0300; pooled=0.1160
[DL GPU] run_id=dl__cnn_room__fusion__lovo__ebl-session__s44__5ccebc peak_cuda_memory_bytes=2685729792
[results upsert] /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/runs.csv: 51 row(s)
[DL seeds] mean +/- std across se

### Analysis Tables

Tables are grouped by evaluation split: Temporal Block, Leave-One-Volunteer-Out (LOVO), Cross Session, and Block versus LOVO (shown only when both splits are available). Every model is reported as **Band CNN** (band-branch architecture) or **Room CNN** (room-stacked architecture, Fusion only), and every metric is the mean +/- SD across the three training seeds.

In [ ]:
# Set to False to save every figure below without rendering it inline (e.g. non-interactive/batch runs).
DISPLAY_PLOTS = True

try:
    all_cnn_predictions = load_all_dl_predictions(
        results_dir,
        bands_to_run=BANDS_TO_RUN,
        split_modes=SPLIT_MODES,
        seeds=SEEDS,
    )
except FileNotFoundError as error:
    all_cnn_predictions = pd.DataFrame()
    print(f"DL predictions unavailable: {error}")

if not all_cnn_predictions.empty:
    cnn_master_table = dl_seed_summary_table(all_cnn_predictions, seeds=SEEDS)
    cnn_per_room_table = dl_per_room_table(all_cnn_predictions, seeds=SEEDS)
else:
    cnn_master_table = pd.DataFrame()
    cnn_per_room_table = pd.DataFrame()

available_splits = set(cnn_master_table["split"].astype(str)) if not cnn_master_table.empty else set()
has_block = "block" in SPLIT_MODES and "block" in available_splits
has_lovo = "lovo" in SPLIT_MODES and "lovo" in available_splits
has_cross_session = (
    "cross_session" in SPLIT_MODES and "cross_session" in available_splits
)

print(
    f"Temporal Block available: {has_block} | LOVO available: {has_lovo} | "
    f"Cross Session available: {has_cross_session}"
)

#### Temporal Block

In [ ]:
if has_block:
    block_master = cnn_master_table.loc[cnn_master_table["split"] == "block"].copy()
    block_master["model"] = dl_display_model_names(block_master["model"])
    display(
        block_master[
            [
                "model",
                "dataset",
                "split",
                "n_seeds",
                "position_accuracy",
                "macro_f1",
                "room_accuracy",
                "mean_distance_error",
                "median_distance_error",
            ]
        ]
    )

    block_rooms = cnn_per_room_table.loc[cnn_per_room_table["split"] == "block"].copy()
    block_rooms["model"] = dl_display_model_names(block_rooms["model"])
    display(block_rooms[["model", "dataset", "split", "true_room", "n_seeds", "position_accuracy"]])
else:
    print("Temporal Block tables skipped: the block split is not selected or unavailable.")

#### Leave-One-Volunteer-Out (LOVO)

In [ ]:
if has_lovo:
    lovo_master = cnn_master_table.loc[cnn_master_table["split"] == "lovo"].copy()
    lovo_master["model"] = dl_display_model_names(lovo_master["model"])
    display(
        lovo_master[
            [
                "model",
                "dataset",
                "split",
                "n_seeds",
                "position_accuracy",
                "macro_f1",
                "room_accuracy",
                "mean_distance_error",
                "median_distance_error",
            ]
        ]
    )

    lovo_rooms = cnn_per_room_table.loc[cnn_per_room_table["split"] == "lovo"].copy()
    lovo_rooms["model"] = dl_display_model_names(lovo_rooms["model"])
    display(lovo_rooms[["model", "dataset", "split", "true_room", "n_seeds", "position_accuracy"]])
else:
    print("LOVO tables skipped: the LOVO split is not selected or unavailable.")

#### Cross Session

In [ ]:
if has_cross_session:
    cross_session_master = cnn_master_table.loc[
        cnn_master_table["split"] == "cross_session"
    ].copy()
    cross_session_master["model"] = dl_display_model_names(cross_session_master["model"])
    display(
        cross_session_master[
            [
                "model",
                "dataset",
                "split",
                "n_seeds",
                "position_accuracy",
                "macro_f1",
                "room_accuracy",
                "mean_distance_error",
                "median_distance_error",
            ]
        ]
    )

    cross_session_rooms = cnn_per_room_table.loc[
        cnn_per_room_table["split"] == "cross_session"
    ].copy()
    cross_session_rooms["model"] = dl_display_model_names(cross_session_rooms["model"])
    display(
        cross_session_rooms[
            ["model", "dataset", "split", "true_room", "n_seeds", "position_accuracy"]
        ]
    )
else:
    print("Cross Session tables skipped: the cross_session split is not selected or unavailable.")

#### Temporal Block versus LOVO

In [ ]:
if has_block and has_lovo:
    comparison_rows = []
    for _, block_row in cnn_master_table.loc[cnn_master_table["split"] == "block"].iterrows():
        lovo_row = cnn_master_table.loc[
            (cnn_master_table["split"] == "lovo")
            & (cnn_master_table["model"] == block_row["model"])
            & (cnn_master_table["dataset"] == block_row["dataset"])
        ]
        if lovo_row.empty:
            continue
        lovo_row = lovo_row.iloc[0]
        comparison_rows.append(
            {
                "model": block_row["model"],
                "dataset": block_row["dataset"],
                "block_position_accuracy_mean": block_row["position_accuracy_mean"],
                "block_position_accuracy_std": block_row["position_accuracy_std"],
                "lovo_position_accuracy_mean": lovo_row["position_accuracy_mean"],
                "lovo_position_accuracy_std": lovo_row["position_accuracy_std"],
                "generalization_gap": (
                    block_row["position_accuracy_mean"] - lovo_row["position_accuracy_mean"]
                ),
            }
        )
    block_vs_lovo_table = (
        pd.DataFrame(comparison_rows).sort_values(["dataset", "model"]).reset_index(drop=True)
    )
    block_vs_lovo_table["model"] = dl_display_model_names(block_vs_lovo_table["model"])
    display(block_vs_lovo_table)
else:
    print("Block vs LOVO comparison table skipped: both splits are required.")

### Temporal Block Plots

In [ ]:
if has_block:
    plot_dl_cdf_comparison(
        all_cnn_predictions,
        split="block",
        seeds=SEEDS,
        band_order=BANDS_TO_RUN,
        save_path=plots_dir / "cdf_comparison_band-vs-room_block.pdf",
        show=DISPLAY_PLOTS,
    )
else:
    print("Temporal Block plots skipped: the block split is not selected or unavailable.")

### LOVO Plots

In [ ]:
if has_lovo:
    plot_dl_cdf_comparison(
        all_cnn_predictions,
        split="lovo",
        seeds=SEEDS,
        band_order=BANDS_TO_RUN,
        save_path=plots_dir / "cdf_comparison_band-vs-room_lovo.pdf",
        show=DISPLAY_PLOTS,
    )

    plot_dl_volunteer_variability(
        all_cnn_predictions,
        seeds=SEEDS,
        band_order=BANDS_TO_RUN,
        save_path=plots_dir / "lovo_volunteer_variability_band-vs-room.pdf",
        show=DISPLAY_PLOTS,
    )
else:
    print("LOVO plots skipped: the LOVO split is not selected or unavailable.")

### Cross Session Plots

In [ ]:
if has_cross_session:
    plot_dl_cdf_comparison(
        all_cnn_predictions,
        split="cross_session",
        seeds=SEEDS,
        band_order=BANDS_TO_RUN,
        save_path=plots_dir / "cdf_comparison_band-vs-room_cross_session.pdf",
        show=DISPLAY_PLOTS,
    )
else:
    print("Cross Session plots skipped: the cross_session split is not selected or unavailable.")

### Temporal Block versus LOVO Plots

In [ ]:
if has_block and has_lovo:
    plot_dl_block_vs_lovo_metrics(
        cnn_master_table,
        band_order=BANDS_TO_RUN,
        save_path=plots_dir / "block_vs_lovo_band-vs-room_metrics.pdf",
        show=DISPLAY_PLOTS,
    )

    block_fusion_cnn = all_cnn_predictions.loc[
        (all_cnn_predictions["split_mode"] == "block")
        & (all_cnn_predictions["model"] == "CNN")
        & (all_cnn_predictions["dataset"] == "Fusion")
    ]
    lovo_fusion_cnn = all_cnn_predictions.loc[
        (all_cnn_predictions["split_mode"] == "lovo")
        & (all_cnn_predictions["model"] == "CNN")
        & (all_cnn_predictions["dataset"] == "Fusion")
    ]
    plot_dl_spatial_generalization(
        block_fusion_cnn,
        lovo_fusion_cnn,
        seeds=SEEDS,
        save_path=plots_dir / "floor_plan_block_vs_lovo_band-cnn_fusion.pdf",
        show=DISPLAY_PLOTS,
    )
else:
    print("Temporal Block vs LOVO plots skipped: both splits are required.")